# Технологии программирования, БИ

## НИУ ВШЭ, 2025-26 учебный год

# Семинар 6. От кода в ноутбуке до собственного образа в Docker Hub

### Цели семинара

1. Поставить и решить небольшую инженерную задачу: написать на Python типовое приложение с классами и бизнес-логикой.
2. Понять, что такое зависимости и зачем нужен `pip`.
3. Научиться переносить код из ноутбука в реальную структуру проекта.
4. Собрать собственный Docker-образ с авторской логикой.
5. Опубликовать код на GitHub (`git push`) и образ на Docker Hub (`docker push`).
6. Обсудить воспроизводимость: что значит «у меня работает», и при чём здесь Docker.
7. Познакомиться с `docker compose` и YAML — оркестрацией нескольких сервисов.

## Что мы делали в прошлый раз

На предыдущем семинаре мы:

- разобрались, что Docker — это про изолированные воспроизводимые окружения;
- работали в GitHub Codespaces как в типовой среде разработчика;
- тянули готовые образы (`docker pull hello-world`, `docker pull nginx`, `docker pull httpd`);
- запускали контейнеры (`docker run`, `docker run -d`, `docker exec -it`);
- собрали два мини-образа (`stats-app`, `temp-app`) — это уже **наша** логика, упакованная в контейнер.

Сегодня делаем следующий логический шаг: пишем не одну функцию, а **настоящий маленький проект** с классами, зависимостями и точкой входа. И этот проект мы доведём до состояния, когда любой человек в мире сможет запустить его одной командой.

## План работы

| Шаг | Что делаем | Кто |
|---|---|---|
| 1 | Ставим задачу: маленький REST API для списка задач | Семинарист |
| 2 | Пишем классы и логику в ноутбуке, проверяем | Вместе |
| 3 | Ставим зависимости (`pip`), обсуждаем что это | Семинарист |
| 4 | Переносим код в `.py` файлы и проверяем на хосте | Вместе |
| 5 | Пишем `Dockerfile`, собираем образ | Семинарист |
| 6 | Запускаем контейнер с пробросом порта | Вместе |
| 7 | `git push` → код на GitHub, `docker push` → образ на Docker Hub | Семинарист |
| 8 | Воспроизводимость: «а у соседа запустится?» | Студенты |
| 9 | `docker compose` и YAML — добавляем второй сервис | Семинарист |
| 10 | Мини-задание | Студенты |

---
# Часть 1. Постановка задачи

Будем писать **простой REST API для списка задач (todo-list)**.

Что должно уметь приложение:
- хранить задачи в памяти;
- отдавать список задач по HTTP (`GET /tasks`);
- добавлять новую задачу (`POST /tasks`);
- помечать задачу выполненной (`POST /tasks/<id>/done`);
- отдавать одну случайную мотивационную цитату через **внешний** API (`GET /quote`) — это даст нам повод подключить вторую зависимость.

На бизнес-логику нам хватит двух классов: `Task` (одна задача) и `TaskManager` (хранилище и операции над ним). Это — типовая схема, которую вы будете встречать постоянно: данные + менеджер.

> **Семинарист:** прежде чем писать код — обсудите со студентами, какие классы они бы выделили сами. Важно, чтобы студенты поняли, что классы — это не «потому что надо», а удобный способ упаковать данные и операции над ними.

---
# Часть 2. Пишем код в ноутбуке

Идея простая: **сначала всё работает в ноутбуке, и только потом мы переносим это в файлы**. Так удобнее отлаживать и нагляднее видеть, что происходит.

## 2.1 Класс `Task`

Это объект-носитель данных: id, текст, дата создания, выполнена ли.

In [1]:
from datetime import datetime

class Task:
    def __init__(self, task_id: int, title: str):
        self.id = task_id
        self.title = title
        self.done = False
        self.created_at = datetime.now()

    def mark_done(self):
        self.done = True

    def to_dict(self) -> dict:
        return {
            "id": self.id,
            "title": self.title,
            "done": self.done,
            "created_at": self.created_at.isoformat(),
        }


t = Task(1, "Прочитать конспект")
print(t.to_dict())
t.mark_done()
print(t.to_dict())

{'id': 1, 'title': 'Прочитать конспект', 'done': False, 'created_at': '2026-05-27T02:35:07.444492'}
{'id': 1, 'title': 'Прочитать конспект', 'done': True, 'created_at': '2026-05-27T02:35:07.444492'}


## 2.2 Класс `TaskManager`

Это «умное хранилище»: умеет добавлять, отдавать список, искать по id, помечать выполненной.

In [2]:
class TaskManager:
    def __init__(self):
        self._tasks: dict[int, Task] = {}
        self._next_id = 1

    def add(self, title: str) -> Task:
        task = Task(self._next_id, title)
        self._tasks[self._next_id] = task
        self._next_id += 1
        return task

    def list_all(self) -> list[dict]:
        return [t.to_dict() for t in self._tasks.values()]

    def mark_done(self, task_id: int) -> Task | None:
        task = self._tasks.get(task_id)
        if task is None:
            return None
        task.mark_done()
        return task


manager = TaskManager()
manager.add("Сходить на семинар")
manager.add("Сделать ДЗ по Docker")
manager.mark_done(1)
manager.list_all()

[{'id': 1,
  'title': 'Сходить на семинар',
  'done': True,
  'created_at': '2026-05-27T02:35:32.076706'},
 {'id': 2,
  'title': 'Сделать ДЗ по Docker',
  'done': False,
  'created_at': '2026-05-27T02:35:32.076724'}]

Отлично — бизнес-логика работает. Теперь нам нужно сделать так, чтобы к этому хранилищу можно было обращаться по сети. Для этого нам понадобятся **зависимости**.

## 2.3 Что такое `pip`?

> **Семинарист:** сначала спросите студентов. «Когда вы пишете `pip install ...` — что это за сущность? Откуда она вообще взялась?»

**`pip`** — это менеджер пакетов для Python. Если совсем по-простому:

- **Пакет (или модуль)** — это чужой Python-код, который кто-то уже написал и опубликовал, чтобы мы не писали то же самое заново. Например, чтобы поднять HTTP-сервер, нам не надо реализовывать TCP-сокеты руками — есть готовый `flask`.
- **PyPI** (Python Package Index, [pypi.org](https://pypi.org)) — это централизованный реестр таких пакетов. По сути — публичный «магазин» библиотек.
- **`pip`** — это CLI-утилита, которая ходит в PyPI, скачивает пакеты и кладёт их в окружение Python.

Аналогия из прошлого семинара: **`docker pull` ↔ `pip install`**. Только `docker pull` тянет образы из Docker Hub, а `pip install` тянет Python-пакеты из PyPI.

Полезные команды:

- `pip install <package>` — установить пакет;
- `pip install <package>==1.2.3` — установить конкретную версию (это важно для воспроизводимости!);
- `pip list` — посмотреть, что уже установлено;
- `pip freeze` — то же самое, но в формате, который годится для `requirements.txt`;
- `pip uninstall <package>` — удалить.


## 2.4 Ставим зависимости

Нам понадобятся **разные** пакеты — это специально, чтобы увидеть, как ведёт себя `pip`:

- `flask` — лёгкий веб-фреймворк, на нём построим API;
- `requests` — клиент HTTP, нужен для похода во внешний API за цитатами.

Можно ставить из ноутбука (пригодится в Codespaces / Colab):

In [ ]:
!pip install flask requests

In [ ]:
!pip list | grep -E "Flask|requests"

> **Семинарист:** обратите внимание студентов на то, что `pip` потянул ещё кучу пакетов «в нагрузку» — это **транзитивные зависимости**. Flask тянет `Werkzeug`, `Jinja2`, `click` и т.д. Именно поэтому в реальных проектах никто не помнит зависимости в голове — их фиксируют в `requirements.txt`.

## 2.5 Простая проверка, что Flask работает

Импортируем `flask` — если импорт прошёл без ошибок, значит пакет действительно установлен в наше окружение. (Каждый раз, когда мы пишем `import`, мы фактически проверяем доступность пакета.)

In [3]:
import flask
import requests
print("flask:", flask.__version__)
print("requests:", requests.__version__)

flask: 3.1.0
requests: 2.32.4


/var/folders/_r/h4zgs51n1dq4hdtkvsl3026h0000gn/T/ipykernel_13898/2386761304.py:3: DeprecationWarning: The '__version__' attribute is deprecated and will be removed in Flask 3.1. Use feature detection or 'importlib.metadata.version("flask")' instead.
  print("flask:", flask.__version__)


Проверим, что `requests` действительно умеет ходить наружу:

In [12]:
import requests

resp = requests.get("https://yandex.ru")
print(resp.status_code)
print(resp.text)

200
<body></body><script nonce='7a9c32a369a607db79d74cdd55293aac'>var it = {"host":"https:\u002F\u002Fsso.dzen.ru\u002Finstall?uuid=1000591d-cb32-4131-9760-e882194e4de5","retpath":"https:\u002F\u002Fdzen.ru\u002F?yredirect=true&is_autologin_ya=true","dzen":"1"};(function() {  function _emitProbe(targetOrigin, step) { try { if (!navigator.sendBeacon) { return; } var body = 'outcome=ready&step=' + step; var blob = new Blob([body], { type: 'application/x-www-form-urlencoded' }); navigator.sendBeacon(targetOrigin + '/beacon/report', blob); } catch (e) {} } function _runBlockingProbe() { _emitProbe('https://' + it.root, 'push_probe_same_site'); try { var installOrigin = new URL(it.host).origin; if (installOrigin && installOrigin !== 'https://' + it.root) { _emitProbe(installOrigin, 'push_probe_cross_site'); } } catch (e) {} } var form = document.createElement('form'); var element1 = document.createElement('input'); var element2 = document.createElement('input'); element1.name = 'retpath'; e

Запускать полноценный Flask-сервер прямо из ноутбука — неудобно (он заблокирует ячейку). Поэтому сразу переходим к переносу кода в файлы. Это и есть «настоящая разработка».

---
# Часть 3. Переносим код в файлы проекта

Спланируем структуру:

```
task_manager/
├── app/
│   ├── __init__.py
│   ├── models.py        # классы Task, TaskManager
│   └── server.py        # Flask-приложение
├── requirements.txt     # зависимости
├── Dockerfile           # рецепт образа
├── docker-compose.yml   # появится позже
├── .gitignore
└── README.md
```

> **Семинарист:** делает всё это в терминале **на глазах у студентов**. Студенты повторяют у себя.

## 3.1 Создаём структуру

In [ ]:
mkdir -p task_manager/app
cd task_manager
touch app/__init__.py app/models.py app/server.py
touch requirements.txt Dockerfile .gitignore README.md .dockerignore
ls -la
ls -la app

## 3.2 Содержимое `app/models.py`

Берём наши классы из ноутбука и кладём их в файл **один в один**. Никаких изменений — просто перенос.

In [ ]:
# app/models.py
from datetime import datetime


class Task:
    def __init__(self, task_id: int, title: str):
        self.id = task_id
        self.title = title
        self.done = False
        self.created_at = datetime.now()

    def mark_done(self) -> None:
        self.done = True

    def to_dict(self) -> dict:
        return {
            "id": self.id,
            "title": self.title,
            "done": self.done,
            "created_at": self.created_at.isoformat(),
        }


class TaskManager:
    def __init__(self):
        self._tasks: dict[int, Task] = {}
        self._next_id = 1

    def add(self, title: str) -> Task:
        task = Task(self._next_id, title)
        self._tasks[self._next_id] = task
        self._next_id += 1
        return task

    def list_all(self) -> list[dict]:
        return [t.to_dict() for t in self._tasks.values()]

    def mark_done(self, task_id: int) -> Task | None:
        task = self._tasks.get(task_id)
        if task is None:
            return None
        task.mark_done()
        return task

### Проверим, что модуль вообще импортируется

Это важный момент: **каждый раз после переноса кода — обязательно импортируем** и убеждаемся, что всё нашлось.

In [ ]:
cd task_manager
python -c "from app.models import Task, TaskManager; m = TaskManager(); m.add('test'); print(m.list_all())"

## 3.3 Содержимое `app/server.py`

А теперь добавим вокруг наших классов веб-обёртку. **Обратите внимание на импорт** — мы тащим `TaskManager` из соседнего файла.

In [ ]:
# app/server.py
import os
import requests
from flask import Flask, request, jsonify

from app.models import TaskManager


app = Flask(__name__)
manager = TaskManager()


@app.get("/")
def root():
    return {"service": "task-manager", "endpoints": ["/tasks"]}


@app.get("/tasks")
def list_tasks():
    return jsonify(manager.list_all())


@app.post("/tasks")
def create_task():
    data = request.get_json(force=True)
    title = data.get("title")
    if not title:
        return {"error": "title is required"}, 400
    task = manager.add(title)
    return task.to_dict(), 201


@app.post("/tasks/<int:task_id>/done")
def finish_task(task_id: int):
    task = manager.mark_done(task_id)
    if task is None:
        return {"error": "not found"}, 404
    return task.to_dict()


if __name__ == "__main__":
    port = int(os.environ.get("PORT", 5009))
    app.run(host="0.0.0.0", port=port)

## 3.4 Файл `requirements.txt`

Это «контракт» проекта: какие пакеты ему нужны. Это первый шаг к воспроизводимости. Без него нельзя — иначе другой человек не знает, что ему ставить.

In [ ]:
# requirements.txt
flask==3.0.3
requests==2.32.3

> **Семинарист:** обсудите со студентами, почему мы **фиксируем версии**. Если просто написать `flask`, то завтра выйдет новая мажорная версия, и проект может перестать собираться. Это классическая причина «вчера работало, сегодня нет».

## 3.5 Проверяем приложение на хосте (КЛЮЧЕВОЙ ЧЕКПОИНТ)

> **Это важнейший этап.** Прежде чем заворачивать что-то в Docker — мы обязаны убедиться, что оно работает на голой машине. Если на хосте не работает — в контейнере точно не заработает.

In [ ]:
cd task_manager
pip install -r requirements.txt

In [ ]:
python -m app.server

Должна появиться надпись вроде:
```
 * Running on http://0.0.0.0:5009
```

В **другом** терминале (или новой вкладке) проверяем:

In [ ]:
curl http://localhost:5009/
curl -X POST http://localhost:5009/tasks -H "Content-Type: application/json" -d '{"title": "learn Docker"}'
curl -X POST http://localhost:5009/tasks -H "Content-Type: application/json" -d '{"title": "do homework"}'
curl http://localhost:5009/tasks
curl -X POST http://localhost:5009/tasks/1/done

Если всё ок — останавливаем сервер `Ctrl+C` и идём контейнеризовать.

> ✅ **Чекпоинт:** на хосте работает → можно переносить в воспроизводимый контейнер.

---
# Часть 4. Заворачиваем в Docker

## 4.1 Пишем Dockerfile

На прошлом семинаре наш `Dockerfile` был совсем простым (три строки). Сейчас понадобятся новые инструкции. Разберём их.

In [ ]:
# Dockerfile
FROM python:3.12-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app/ ./app/

ENV PORT=5009
EXPOSE 5009

CMD ["python", "-m", "app.server"]

### Что нового по сравнению с прошлым семинаром

- **`WORKDIR /app`** — задаёт рабочую директорию внутри образа. Все последующие `COPY`, `RUN`, `CMD` исполняются относительно неё. Это аналог `cd /app`, но «навсегда» для этого образа.
- **`COPY requirements.txt .`** — копируем **сначала только** файл зависимостей, а не весь проект. Зачем такая хитрость?
- **`RUN pip install --no-cache-dir -r requirements.txt`** — `RUN` исполняет команду в момент сборки. Этот слой будет закэширован, и при следующих сборках (если `requirements.txt` не менялся) Docker не будет переустанавливать зависимости заново. Это **слоистая модель Docker** в действии.
- **`COPY app/ ./app/`** — только теперь копируем код. Если поменяется только код (а зависимости нет), Docker пересоберёт только последние слои — это в разы быстрее.
- **`ENV PORT=5009`** — переменная окружения внутри образа.
- **`EXPOSE 5009`** — декларативное «приложение слушает 5009-й порт». Это документация для людей и инструментов; саму публикацию порта делает `docker run -p`.
- **`CMD ["python", "-m", "app.server"]`** — команда по умолчанию, которая выполнится при запуске контейнера.

> **Семинарист:** обязательно проговорите идею **порядка слоёв**. Это классический инженерный приём, и студенты должны его прочувствовать. Если поменять порядок (сначала `COPY app/`, потом `requirements.txt`) — кэш будет сбрасываться при каждой правке кода, и сборка станет в 10× дольше.

## 4.2 `.gitignore` и `.dockerignore`

Чтобы не таскать в репозиторий и в образ ненужное:

In [ ]:
# .gitignore
__pycache__/
*.pyc
.venv/
.env
.DS_Store
.idea/
.vscode/

In [ ]:
# .dockerignore
__pycache__/
*.pyc
.venv/
.git/
.gitignore
Dockerfile
docker-compose.yml
README.md

## 4.3 Собираем образ

In [ ]:
cd task_manager
docker build -t task-manager:0.1 .

In [ ]:
docker images | grep task-manager

## 4.4 Запускаем контейнер

Тут впервые появится **новый флаг** — проброс порта:

In [ ]:
docker run -d --name tm -p 5009:5009 task-manager:0.1

- `-p 5009:5009` — `<хост>:<контейнер>`. Внутри контейнера приложение слушает 5009-й порт; мы прокидываем его на 5009-й порт хоста.
- `--name tm` — удобное имя.
- `-d` — в фоне.

Проверяем:

In [ ]:
docker ps
docker logs tm
curl http://localhost:5009/
curl -X POST http://localhost:5009/tasks -H "Content-Type: application/json" -d '{"title":"task inside container!"}'
curl http://localhost:5009/tasks
docker logs tm

SyntaxError: invalid syntax (1442690086.py, line 1)

Работает! Останавливать пока не будем — пригодится дальше.

> **Семинарист:** ВАЖНО: обратите внимание студентов, что на хосте мы **ничего не устанавливали** (Flask, requests). Установка случилась только внутри образа. Это и есть суперсила Docker — изоляция.

---
# Часть 5. Воспроизводимость — попробуем «передать» проект соседу

Сейчас классический момент, когда обычно ломается: «у меня работает, у тебя не работает».

Без Docker этот шаг — кошмар: разные ОС, разные версии Python, разные пути, разные пакеты. С Docker мы передаём не код «как есть», а **готовый, собранный, фиксированный образ**.

Есть два способа передать образ:

1. через файл (`docker save` / `docker load`) — оффлайн;
2. через **реестр** (`docker push` / `docker pull`) — нормальный, продакшен-способ.

> **Семинарист:** обязательно проговорите, что **«воспроизведётся всегда»** — это идеализация. Реально мешать могут:
> - архитектура процессора (ARM vs x86_64) — здесь нам поможет `--platform` при сборке;
> - объёмы и тома, если они подмонтированы.

---
# Часть 6. Публикуем код и образ

## 6.1 Код на GitHub: `git push`

Базовые команды `git`, которые нам нужны сегодня (большинство студенты уже видели):

> **Семинарист:** следующие команды, как вопрос студентам. Вспоминаем прошлый семинар
> - Показать студентам наглядно как это работает можно. 
> - Студенты пусть **не** прогоняют (займет лишнее время)

In [ ]:
cd task_manager
git init
git add .
git status
git commit -m "Initial commit: task-manager with Flask + Docker"

Создаём пустой репозиторий на GitHub (через веб-интерфейс или `gh repo create`), затем:

In [ ]:
git remote add origin https://github.com/<your-username>/task-manager.git
git branch -M main
git push -u origin main

Теперь сосед может:

In [ ]:
git clone https://github.com/<your-username>/task-manager.git
cd task-manager
# и собрать у себя:
docker build -t task-manager:0.1 .

## 6.2 Образ на Docker Hub: `docker push`

Docker Hub — это «PyPI» / «GitHub» для образов. Заводим аккаунт на [hub.docker.com](https://hub.docker.com), затем:

> **Семинарист:** можно без демонстрации, просто по коду пройтись

In [ ]:
docker login

Чтобы запушить — образу нужно дать имя в формате `<username>/<repo>:<tag>`. Локальный тег `task-manager:0.1` для этого не подходит, его надо **переименовать**:

In [ ]:
docker tag task-manager:0.1 <your-dockerhub-username>/task-manager:0.1
docker images | grep task-manager

In [ ]:
docker push <your-dockerhub-username>/task-manager:0.1

### Проверка воспроизводимости

Теперь любой человек в мире может:

In [ ]:
docker pull <your-dockerhub-username>/task-manager:0.1
docker run -d -p 5009:5009 <your-dockerhub-username>/task-manager:0.1
curl http://localhost:5009/

---
## Маленькое отступление: это лишь частный случай

Мы сделали один сервис на Python и упаковали его. В реальной системе обычно работает **много** сервисов:

- веб-приложение (как наше);
- база данных (PostgreSQL, MySQL);
- кэш (Redis);
- очередь сообщений (RabbitMQ, Kafka);
- фронтенд (React);
- nginx как реверс-прокси;
- и так далее.

Каждый из них может быть на своём языке (Python, Go, Java, JS, C++...), и запускать их вручную — мучение. Поэтому появляется следующий уровень — **оркестрация контейнеров**. Самая простая её форма, с которой все начинают, — это `docker compose`.

---
# Часть 7. Docker Compose и YAML

## 7.1 Что такое YAML

YAML (Yet Another Markup Language) — формат конфигурационных файлов. Сравните JSON и YAML:

```json
{
  "name": "tm",
  "port": 5009,
  "tags": ["web", "python"]
}
```

```yaml
name: tm
port: 5009
tags:
  - web
  - python
```

Ключевое: **отступы значимы** (как в Python), вложенность задаётся пробелами, табы запрещены.

## 7.2 Что такое Docker Compose

`docker compose` — это инструмент, который читает `docker-compose.yml` и одной командой поднимает **набор связанных сервисов**.

Вместо длинной команды `docker run -d --name tm -p 5009:5009 -e PORT=5009 task-manager:0.1` мы описываем то же самое декларативно в файле.

## 7.3 Минимальный `docker-compose.yml`

Начнём с одного сервиса — нашего:

In [ ]:
# docker-compose.yml
services:
  api:
    build: .
    image: task-manager:0.1
    ports:
      - "5009:5009"
    environment:
      PORT: 5009
    restart: unless-stopped

In [ ]:
cd task_manager
touch docker-compose.yml

Перед тем как запускать — остановим старый контейнер, чтобы порт 5009 не был занят:

In [ ]:
docker stop tm && docker rm tm

In [ ]:
docker compose up -d
docker compose ps
curl http://localhost:5009/

### Основные команды compose

- `docker compose up` — поднять всё, что описано в файле; `-d` — в фоне;
- `docker compose down` — остановить и удалить контейнеры;
- `docker compose ps` — статус сервисов;
- `docker compose logs -f` — смотреть логи;
- `docker compose build` — пересобрать;
- `docker compose restart api` — перезапустить конкретный сервис.

## 7.4 Добавляем второй сервис

Чтобы прочувствовать, **зачем вообще нужен compose**, добавим в нашу систему второй сервис — Redis (in-memory key-value хранилище). Пусть наш API сможет хранить там счётчик обращений.

Сначала обновим `requirements.txt` — добавим Python-клиент к Redis:

In [ ]:
# requirements.txt
flask==3.0.3
requests==2.32.3
redis==5.0.7

Добавим в `app/server.py` новый эндпоинт `/hits`:

In [ ]:
# фрагмент app/server.py — добавить в начало
import redis
redis_client = redis.Redis(
    host=os.environ.get("REDIS_HOST", "localhost"),
    port=int(os.environ.get("REDIS_PORT", 6379)),
    decode_responses=True,
)

@app.get("/hits")
def hits():
    n = redis_client.incr("hits")
    return {"hits": n}

Обновим `docker-compose.yml`:

In [ ]:
# docker-compose.yml
services:
  api:
    build: .
    image: task-manager:0.2
    ports:
      - "5009:5009"
    environment:
      PORT: 5009
      REDIS_HOST: redis
      REDIS_PORT: 6379
    depends_on:
      - redis
    restart: unless-stopped

  redis:
    image: redis:7-alpine
    ports:
      - "6379:6379"
    restart: unless-stopped

**Ключевая магия:** в `REDIS_HOST: redis` мы пишем не IP-адрес, а **имя сервиса**. Compose сам создаёт сеть, в которой сервисы видят друг друга по именам. Это и есть оркестрация в простейшем виде.

Запускаем:

In [ ]:
docker compose down
docker compose up -d --build
docker compose ps
curl http://localhost:5009/hits
curl http://localhost:5009/hits
curl http://localhost:5009/hits

Счётчик растёт. Поднимем и опустим всё одной командой — это и есть смысл `compose`.

In [ ]:
docker compose down

---
# Что мы сегодня сделали

- Поставили реальную задачу и поэтапно её решили.
- Написали типовое приложение с классами `Task` и `TaskManager`.
- Подключили внешние зависимости через `pip` и поняли, что это вообще такое.
- Перенесли код из ноутбука в нормальную структуру проекта.
- Завернули проект в Docker-образ с правильным послойным кэшированием.
- Опубликовали код на GitHub и образ на Docker Hub.
- Обсудили воспроизводимость и её ограничения.
- Подняли многосервисную систему через `docker compose`.

Ключевая мысль: **код → артефакт → реестр → запуск где угодно**. Это базовая цепочка современной разработки, и теперь у вас есть свой собственный пример, который вы прошли руками.